In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "m5")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
spark.sql(f"USE {CATALOG}.{SCHEMA}")
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/raw"

In [0]:
from pyspark.sql import DataFrame, functions as F

ID_COLS = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

def melt_sales(sales: DataFrame) -> DataFrame:
    """Wide (one column per day) -> long (one row per item, store, day)."""
    day_cols = [c for c in sales.columns if c.startswith("d_")]
    return sales.select(*ID_COLS, *day_cols).unpivot(
        ids=ID_COLS, values=day_cols,
        variableColumnName="d", valueColumnName="units")

def attach_calendar_and_prices(long_sales, calendar, prices) -> DataFrame:
    cal = calendar.select(
        "d", F.to_date("date").alias("date"), "wm_yr_wk",
        "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI")
    pr = prices.select("store_id", "item_id", "wm_yr_wk", "sell_price")
    return (long_sales
            .join(cal, on="d", how="left")
            .join(pr, on=["store_id", "item_id", "wm_yr_wk"], how="left"))